# Calling Reprosyn from a python script

Reprosyn is first a foremost a command line tool. It does not yet have a nice api as a package. 

This notebook describes how to call a reprosyn method `mst`, programmatically varying parameters and saving the output. 

We assume that you have installed reprosyn into whichever python environment you are working in using `pip install git+https://github.com/alan-turing-institute/reprosyn`.

In [ ]:
import pandas as pd
import subprocess
from io import StringIO
import json

In [ ]:
census = pd.read_csv('https://raw.githubusercontent.com/alan-turing-institute/reprosyn/main/src/reprosyn/datasets/2011-census-microdata/2011-census-microdata-small.csv')

census.head()

`Reprosyn` takes input from `STDIN` and outputs to `STDOUT`, unless specifically given filepaths.

This means we can loop easily using subprocess. To expedite the example we fix the size of the datasets generated to `10`.

In [ ]:
size = 10
epsilon = [1,10,100]
command='mst'
inp = bytes(census.to_csv(), 'utf-8')

outputs = {}
for e in epsilon:
    print(f"running {command} for epsilon = {e}")
    out = subprocess.run(["rsyn", "--size", f"{size}", command, "--epsilon", f"{e}"], input=inp, capture_output=True)
    print('stderr: ', out.stderr)
    df = pd.read_csv(StringIO(cp.stdout.decode()))
    outputs[e] = df


In [ ]:
outputs

### Passing a json config.

To make customisation easy you can pass a config file to `reprosyn`. 

First, we use the `--generateconfig` to retrieve a file with standard defaults that we can edit.


In [ ]:
out = subprocess.run(["rsyn", "--generateconfig", command], capture_output=True)

In [ ]:
config = json.load(StringIO(out.stdout.decode()))
print(config)

Note that `--size` is not an option. This is because `--size` is a global parameter, not a method configuration.

We can use the global parameter `--configstring` to amend the method default values with the config file.


In [ ]:
dicts = {}
for e in epsilon:
    config['epsilon'] = e
    out = subprocess.run(["rsyn", "--size", f"{size}", "--configstring", f"{json.dumps(config)}", command], input=inp, capture_output=True)
    print('stderr: ', out.stderr)
    df = pd.read_csv(StringIO(cp.stdout.decode()))
    dicts[e] = df

In [ ]:
dicts